In [0]:
display(
    spark.sql("""
        SELECT
            AddedDate,
            COUNT(*) AS total_events,
            COUNT(DISTINCT CountryCode) AS countries
        FROM workspace.gold.country_ingestion_daily
        GROUP BY AddedDate
        ORDER BY AddedDate
    """)
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

country_window = (
    Window
    .partitionBy("CountryCode")
    .orderBy("AddedDate")
)

country_rolling_window = (
    country_window
    .rowsBetween(-6, -1)
)

country_features = (
    spark.table("workspace.gold.country_ingestion_daily")

    .withColumn(
        "RollingMean7",
        F.avg("EventCount").over(country_rolling_window)
    )

    .withColumn(
        "RollingStd7",
        F.stddev("EventCount").over(country_rolling_window)
    )

    .withColumn(
        "PreviousEventCount",
        F.lag("EventCount", 1).over(country_window)
    )

    .withColumn(
        "EventCountChange",
        F.col("EventCount") -
        F.col("PreviousEventCount")
    )

    .withColumn(
        "EventCountRatio",
        F.when(
            F.col("PreviousEventCount") > 0,
            F.col("EventCount") /
            F.col("PreviousEventCount")
        )
    )

    .withColumn(
        "DeviationFromRollingMean",
        F.when(
            F.col("RollingMean7").isNotNull(),
            F.col("EventCount") -
            F.col("RollingMean7")
        )
    )

    .withColumn(
        "RollingZScore",
        F.when(
            F.col("RollingStd7") > 0,
            (
                F.col("EventCount") -
                F.col("RollingMean7")
            ) / F.col("RollingStd7")
        )
    )
)

display(
    country_features
    .orderBy("AddedDate", F.col("EventCount").desc())
    .limit(50)
)

In [0]:
display(
    country_features.select(
        "AddedDate",
        "CountryCode",
        "EventCount",
        "RollingMean7",
        "RollingStd7",
        "RollingZScore"
    )
    .filter(F.col("RollingMean7").isNotNull())
    .orderBy("AddedDate")
    .limit(50)
)

In [0]:
print(
    "Feature rows:",
    country_features.filter(
        F.col("RollingMean7").isNotNull()
    ).count()
)